In [ ]:
EXPECTED_PARROT_API_KEY = "api_key"

In [ ]:
import os
import pandas as pd
import concurrent.futures
from datetime import datetime
from edsl import QuestionNumerical, Scenario, ScenarioList, Model, ModelList, Survey

# Create a directory to store the results
results_dir = "fed_treatment_results"
os.makedirs(results_dir, exist_ok=True)

# Create a multiple choice question about Federal Reserve policy
q3 = QuestionNumerical(
    question_name = "Q1_S",
    question_text = """
    Consider the following current event: {{ info }}
    What do you expect the rate of inflation to be over the next 12 months? Please give your best guess. 
    """,
    # min_value = 1
    # max_value = 100 
)
q4 = QuestionNumerical(
    question_name = "Q2_L",
    question_text = """
    Consider the following current event: {{ info }}
    What do you expect the rate of inflation to be over the 12-month period beginning 24 months from now
    and ending 36 months from now? 
    Please give your best guess.
    """,
    # min_value = 1, 
    # max_value = 100 
)

# Create scenarios from your treatments
# Treatments including control
treatments = [
    # Control (no information)
    {
        "treatment": "T_0",
        "description": "Control with no information",
        "info": ""
    },
    # a. Changes in Language Complexity
    # Neutral Information
    {
        "treatment": "T_a1",
        "description": "Neutral - Simplified language",
        "info": (
            "The Federal Reserve monitors economic data like employment figures, prices, "
            "and economic growth to make decisions about interest rates. The current "
            "Federal Funds Rate is 4.25%-4.5%."
        )
    },
    {
        "treatment": "T_a2",
        "description": "Neutral - Technical language",
        "info": (
            "The Federal Open Market Committee utilizes a range of economic indicators "
            "including labor market conditions, inflation pressures, inflation expectations, "
            "and financial developments to calibrate monetary policy. The target range "
            "for the federal funds rate is currently 4.25 to 4.50 percent."
        )
    },
    # Anti-Recession (Accommodative Policy)
    {
        "treatment": "T_a3",
        "description": "Anti-recession - Simplified language",
        "info": (
            "The Federal Reserve is lowering interest rates to help boost the economy. "
            "This makes it cheaper for people and businesses to borrow money, which can "
            "create more jobs and economic activity."
        )
    },
    {
        "treatment": "T_a4",
        "description": "Anti-recession - Technical language",
        "info": (
            "The Federal Open Market Committee is implementing accommodative monetary policy "
            "by reducing the target range for the federal funds rate to stimulate aggregate demand. "
            "This policy adjustment is intended to facilitate credit accessibility, promote employment "
            "growth, and support economic expansion."
        )
    },
    # Anti-Inflation (Restrictive Policy)
    {
        "treatment": "T_a5",
        "description": "Anti-inflation - Simplified language",
        "info": (
            "The Federal Reserve is raising interest rates to help bring down high prices. "
            "Higher interest rates make borrowing more expensive, which slows down spending "
            "and helps control rising costs."
        )
    },
    {
        "treatment": "T_a6",
        "description": "Anti-inflation - Technical language",
        "info": (
            "The Federal Open Market Committee is implementing contractionary monetary policy "
            "by increasing the target range for the federal funds rate to counter inflationary pressures. "
            "This policy stance is designed to moderate demand, restore price stability, and anchor "
            "inflation expectations at levels consistent with the Committee's 2 percent objective."
        )
    },
    # b. Framing of Policy Commitments
    # Neutral Information
    {
        "treatment": "T_b1",
        "description": "Neutral - Conditional statement",
        "info": (
            "The Federal Reserve will adjust interest rates based on incoming economic data. "
            "Future policy decisions will depend on developments in employment, inflation, and "
            "broader economic conditions."
        )
    },
    {
        "treatment": "T_b2",
        "description": "Neutral - Unconditional statement",
        "info": (
            "The Federal Reserve will hold its next policy meeting on June 17-18. The committee "
            "will issue its regular statement and economic projections following the conclusion "
            "of the meeting."
        )
    },
    # Anti-Recession (Accommodative Policy)
    {
        "treatment": "T_b3",
        "description": "Anti-recession - Conditional statement",
        "info": (
            "The Federal Reserve will consider cutting interest rates if economic growth continues "
            "to slow and unemployment rises above 4.5%. The committee remains prepared to ease "
            "monetary policy if risks to economic activity increase."
        )
    },
    {
        "treatment": "T_b4",
        "description": "Anti-recession - Unconditional statement",
        "info": (
            "The Federal Reserve will reduce the federal funds rate by 0.25 percentage points at its "
            "next meeting and plans further cuts totaling 0.75 percentage points by the end of the year "
            "to support economic growth."
        )
    },
    # Anti-Inflation (Restrictive Policy)
    {
        "treatment": "T_b5",
        "description": "Anti-inflation - Conditional statement",
        "info": (
            "The Federal Reserve will consider additional rate increases if inflation remains elevated "
            "and fails to show substantial progress toward the 2% target. The committee stands ready "
            "to tighten policy further as warranted by the data."
        )
    },
    {
        "treatment": "T_b6",
        "description": "Anti-inflation - Unconditional statement",
        "info": (
            "The Federal Reserve will maintain high interest rates throughout 2025. The committee will "
            "not reduce rates until it has gained complete confidence that inflation is returning to the 2% "
            "target sustainably."
        )
    },
    # c. Time Horizons of Guidance
    # Neutral Information
    {
        "treatment": "T_c1",
        "description": "Neutral - Short-term horizon",
        "info": (
            "The Federal Reserve will assess incoming data over the next six weeks before its June meeting. "
            "The committee will consider the most recent inflation readings and employment report when "
            "making its next policy decision."
        )
    },
    {
        "treatment": "T_c2",
        "description": "Neutral - Long-term horizon",
        "info": (
            "The Federal Reserve's long-term goals include price stability and maximum sustainable employment. "
            "Over the coming years, the committee aims to conduct monetary policy that achieves inflation "
            "averaging 2% over time."
        )
    },
    # Anti-Recession (Accommodative Policy)
    {
        "treatment": "T_c3",
        "description": "Anti-recession - Short-term horizon",
        "info": (
            "The Federal Reserve is focused on improving economic conditions in the near term. In the next "
            "three months, the committee will prioritize actions that can quickly boost employment and "
            "economic activity."
        )
    },
    {
        "treatment": "T_c4",
        "description": "Anti-recession - Long-term horizon",
        "info": (
            "The Federal Reserve is launching a multi-year accommodative policy approach. The committee plans "
            "to maintain lower interest rates throughout the next two years to ensure a durable economic recovery "
            "and return to maximum employment."
        )
    },
    # Anti-Inflation (Restrictive Policy)
    {
        "treatment": "T_c5",
        "description": "Anti-inflation - Short-term horizon",
        "info": (
            "The Federal Reserve will maintain its current restrictive stance for the next three months. The committee "
            "expects to see meaningful progress on inflation reduction within this quarter before considering any policy adjustments."
        )
    },
    {
        "treatment": "T_c6",
        "description": "Anti-inflation - Long-term horizon",
        "info": (
            "The Federal Reserve is committed to a sustained campaign against inflation over the next several years. The committee "
            "anticipates that returning inflation to the 2% target will require maintaining restrictive policy well into 2026."
        )
    }
]

# Function to run a single iteration of the experiment
def run_single_experiment(iteration_number):
    try:
        # Create scenario list from treatments
        s = ScenarioList(Scenario(treatment) for treatment in treatments)
        
        # Create survey
        survey = Survey(questions=[q3, q4])
        
        # Specify the models
        models = ModelList([
            Model("gpt-4.1-2025-04-14", service_name="openai", temperature=1),
            Model("claude-3-7-sonnet-20250219", service_name="anthropic"),
            Model("meta-llama/Meta-Llama-3-70B-Instruct", service_name="deep_infra"),
        ])
        
        # Run the experiment
        results = survey.by(s).by(models).run()
        
        # Generate a timestamp for the filename
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        # Make sure the iteration number has a consistent number of digits (padding with zeros)
        padded_iteration = str(iteration_number).zfill(3)
        filename = f"fed_treatment_results_run_{padded_iteration}_{timestamp}.csv"
        filepath = os.path.join(results_dir, filename)
        
        # Save results to CSV
        results.to_csv(filepath)
        
        return filepath
    except Exception as e:
        print(f"Error in run {iteration_number}: {str(e)}")
        raise e  # Re-raise the exception to be caught by the executor

# Function to merge all CSV files into one
def merge_all_results(file_paths):
    print("Merging all results...")
    
    # Initialize an empty list to store all dataframes
    all_dfs = []
    
    # Read each CSV file and add a run_id column
    for i, file_path in enumerate(file_paths):
        df = pd.read_csv(file_path)
        df['run_id'] = i + 1  # Add a run_id column (1-indexed)
        all_dfs.append(df)
    
    # Concatenate all dataframes
    merged_df = pd.concat(all_dfs, ignore_index=True)
    
    # Save the merged dataframe
    merged_filepath = os.path.join(results_dir, "fed_treatment_all_results.csv")
    merged_df.to_csv(merged_filepath, index=False)
    
    print(f"All results merged and saved to {merged_filepath}")
    
    return merged_filepath

# Main execution
def main():
    # Run the experiment 100 times in parallel
    file_paths = []
    
    max_workers = None  
    
    print(f"Starting 100 experiment runs in parallel...")
    
    # Using ThreadPoolExecutor to run jobs in parallel
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all 100 runs to the executor
        future_to_run = {executor.submit(run_single_experiment, i): i for i in range(1, 101)}
        
        # Process results as they complete
        for future in concurrent.futures.as_completed(future_to_run):
            run_id = future_to_run[future]
            try:
                filepath = future.result()
                file_paths.append(filepath)
                print(f"Run {run_id} completed successfully")
            except Exception as e:
                print(f"Run {run_id} generated an exception: {e}")
    
    # Sort file_paths by run number to ensure consistent order
    file_paths.sort()
    
    # Merge all results
    merged_filepath = merge_all_results(file_paths)
    
    print(f"Experiment completed successfully with 100 runs.")
    print(f"Individual results are in the '{results_dir}' folder.")
    print(f"Combined results are in '{merged_filepath}'.")

if __name__ == "__main__":
    main()